# Test 02: Supervised and Unsupervised Learning

Name: Lon Cherryholmes, Sr.

Date: Friday 12/05/2025

CWID: xxx17840

You need to get the questions from our class MyLeoOnline site by starting Test 02.  You should be given 2 multi-part questions.

Please make sure that the notebook that you submit for grading runs all cells cleanly from top to bottom if a **Run -> Run All Cells** is performed.  Please give your name and CWID in the provided fields below to ensure your test work is properly attributed.  As a reminder, all work on tests and assignments for this class is individual work unless otherwise noted.  You may not work with others, nor use past student's works or answers created by others for these test questions.

## General Setup

The following imports may be useful for Test 02, and should be sufficient to finish the
questions.  However, if you would like to import additional functions or objects, you may
do so, though they need to come from `scikit-learn` library, or other functions from basic
libraries like `numpy`, `matplotlib` or `pandas`.

In [ ]:
# Basic imports that are generally useful
import numpy as np
import pandas as pd
import seaborn as sbn
import matplotlib.pyplot as plt
import matplotlib
import sklearn

# Imports specific to these test questions
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.datasets import load_wine
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import confusion_matrix
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

# Please put all imports you add for your work here
from google.colab import drive
from pprint import pprint, pformat
from pandas.plotting import scatter_matrix
from sklearn.metrics import ConfusionMatrixDisplay

In [ ]:
# Notebook-wide options to improve appearance of basic matplotlib figures and numpy output
%matplotlib inline
matplotlib.rcParams['figure.figsize'] = (10, 8) # set default figure size, 10in by 8in
np.set_printoptions(suppress=True)
plt.rc('axes', labelsize=14)
plt.rc('xtick', labelsize=12)
plt.rc('ytick', labelsize=12)
plt.rc('figure', titlesize=18)
plt.rc('legend', fontsize=14)
plt.rcParams['figure.figsize'] = (12.0, 8.0) # default figure size if not specified in plot

Start Test 02 in MyLeoOnline to get the test questions you need to answer.  You will need
to use the `wine` dataset.  In addition to this notebook, you should have been given
a `wine.csv` file to download for use on this question.  Please put the file
into the same directory as where you run your notebook and do not use an absolute
path name in your command to load the file into a Pandas dataframe or NumPy array.

In [ ]:
drive.mount('/content/modules', force_remount=True)
wine = pd.read_csv("/content/modules/My Drive/CSCI-574/wine.csv", delimiter=';')

## Data Exploration

In [ ]:
print(wine.info())

In [ ]:
wine['quality'].value_counts()

In [ ]:
display(wine.describe())

In [ ]:
wine.hist(bins=64, figsize=(12,8));

In [ ]:
wine_corr = wine.corr()
wine_corr['quality'].sort_values(ascending=False)

In [ ]:
scatter_matrix(wine[['quality', 'alcohol', 'density', 'chlorides', 'volatile acidity', 'total sulfur dioxide']], figsize=(14,10));

## Prep 1: Supervised Learning

In [ ]:
splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
strat_splits = []
for train_index, test_index in splitter.split(wine, wine['quality']):
    strat_splits.append([wine.iloc[train_index], wine.iloc[test_index]])

In [ ]:
print(wine['quality'].value_counts() / len(wine))
pprint([(w['quality'].value_counts() / len(w)).to_numpy() for w, _ in strat_splits])

In [ ]:
wine_train, wine_test = strat_splits[0]
print(wine_train.shape)
print(wine_test.shape)
X_train = wine_train.iloc[:, 0:11].to_numpy()
y_train = wine_train.iloc[:, 11].to_numpy()
X_test  = wine_test.iloc[:, 0:11].to_numpy()
y_test  = wine_test.iloc[:, 11].to_numpy()
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

In [ ]:
transformer = StandardScaler()
X_train_scaled = transformer.fit_transform(X_train)
X_test_scaled = transformer.fit_transform(X_test)
model = LogisticRegression(penalty='l2')
model.fit(X_train_scaled, y_train)
y_test_pred = model.predict(X_test_scaled)
ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred);

In [ ]:
transformer = StandardScaler()
X_train_scaled = transformer.fit_transform(X_train)
X_test_scaled = transformer.fit_transform(X_test)
model = SVC()
model.fit(X_train_scaled, y_train)
y_test_pred = model.predict(X_test_scaled)
ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred);

In [ ]:
model = ExtraTreesClassifier()
model.fit(X_train, y_train)
y_test_pred = model.predict(X_test)
ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred);

In [ ]:
pipe = Pipeline([
    ('pre', StandardScaler()),
    ('clf', LogisticRegression())
])
pipe.fit(X_train, y_train)
y_test_pred = pipe.predict(X_test)
ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred)
grid_params = [
    { 'clf__solver': ['lbfgs', 'newton-cholesky', 'newton-cg'],
      'clf__penalty': [None, 'l2'],
      'clf__max_iter': [512],
      'clf__class_weight': [None, 'balanced']
    },
    { 'clf__solver': ['liblinear'],
      'clf__penalty': ['l1', 'l2'],
      'clf__max_iter': [512],
      'clf__class_weight': [None, 'balanced']
    },
    { 'clf__solver': ['saga'],
      'clf__penalty': [None, 'l1', 'l2'],
      'clf__max_iter': [1024],
      'clf__class_weight': [None, 'balanced']
    },
    { 'clf__solver': ['saga'],
      'clf__penalty': ['elasticnet'],
      'clf__max_iter': [2048],
      'clf__l1_ratio': [0.0, 0.25, 0.5, 0.75, 1.0],
      'clf__class_weight': [None, 'balanced']
    },
]
grid_search = GridSearchCV(pipe, grid_params, cv=3, scoring='accuracy')
grid_search.fit(X_train, y_train)
grid_search.best_params_

## Question 1: Supervised Learning (65 pts)

In [ ]:
X = wine.iloc[:, 0:11].to_numpy()
y = wine.iloc[:, 11].to_numpy()
print(X.shape)
print(y.shape)

## Prep 2: Unsupervised Learning

In [ ]:
transformer = StandardScaler()
X = wine.iloc[:, 0:11].to_numpy()
y = wine.iloc[:, 11].to_numpy()
X_scaled = transformer.fit_transform(X)
print(X_scaled.shape)

In [ ]:
pca = PCA()
pca.fit(X_scaled)
print(pca.n_components_)
print(pca.components_)
print(pca.explained_variance_)
print(pca.explained_variance_ratio_)

In [ ]:
pca = PCA(n_components=0.75)
Xd = pca.fit_transform(X_scaled)
print(pca.components_)
print(pca.explained_variance_)
print(pca.explained_variance_ratio_)

In [ ]:
k = 7
kmeans = KMeans(n_clusters=k, n_init=3)
y_pred = kmeans.fit_predict(X_scaled)
print(y)
print(y_pred)
print(kmeans.cluster_centers_)

## Question 2: Unsupervised Learning (35 pts)

In [ ]:
X = wine.iloc[:, 0:11].to_numpy()
print(X.shape)